# Known-stream search: example

How to make matched-filter maps of a known stream in the Euclid DR1 × DES Y6 footprint:

1. the stars and the survey footprint;
2. the galstreams streams crossing the footprint, at dm > 14;
3. for one stream: its frame, the window of the maps, its isochrones;
4. the map at one distance modulus;
5. the maps over the grid of distance moduli, with the video and the file of maps;
6. maps that follow the distance of the track;
7. the same in one call, for one stream or for all of them.

The maps are star counts in the match filter, in the stream frame (phi1, phi2). No background is subtracted. There is one set of maps per isochrone.

**Phase 2 stub.** Only the import cell runs: the other cells call functions that are not written yet (they raise `NotImplementedError`). Phase 3 makes this notebook run end to end on a validation stream (Section 7 of `docs/dev/known_streams_context.md`).

In [ ]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.known_stream_search import (
    known_stream_search_dm_along_track,
    known_stream_search_dm_scan,
    known_stream_search_single_dm,
)
from streameucliddes.catalogs.euclid_des_catalog import get_eucliddes_stars
from streameucliddes.catalogs.galstreams_catalog import (
    galstreams_dm_along_track,
    galstreams_phi12_window,
    load_galstreams,
    select_galstreams_in_footprint,
)
from streameucliddes.catalogs.survey_footprint import (
    survey_footprint_healpix_mask,
    survey_footprint_phi12_mask,
)
from streameucliddes.catalogs.isochrone_parameters import known_stream_isochrone_parameters
from streameucliddes.plotting import plot_style
from streameucliddes.plotting.make_videos import make_gif_from_plots, make_mp4
from streameucliddes.plotting.plot_known_stream_search import (
    plot_galstreams_selection,
    plot_known_stream_search_single_dm,
)
from streameucliddes.stream_frames import phi12_bin_centres, radec_to_phi12
from streameucliddes.workflows.run_config import RunConfig
from streameucliddes.workflows.run_known_stream_search import (
    run_all_known_stream_searches,
    run_known_stream_search,
    save_known_stream_search_maps,
)

## 1. Stars and footprint

The Euclid × DES stars. Their magnitudes are already dereddened: there is no dust correction below.

**Until the Euclid DR1 data are available**, `USE_FAKE_CATALOG = True` makes fake stars instead, uniform on the sphere over ra -20 to 100 and dec -70 to 0 (about 6,500 deg², like DES), with DES g and r uniform in colour-magnitude space (g from 17 to 25, g - r from -0.5 to 1.5) and errors of 0.02 mag: a background without any stream, to run the pipeline. That area covers the DES streams of the validation (ATLAS, Phoenix, Turranburra; Shipp et al. 2018) and the windows of their maps.

The footprint comes from the star counts for now: a HEALPix pixel is in the footprint when it holds a star. A mask is True outside the footprint.

In [ ]:
USE_FAKE_CATALOG = True  # no Euclid DR1 data yet; False: the Euclid x DES catalog

if USE_FAKE_CATALOG:
    rng = np.random.default_rng(42)
    N_FAKE_STARS = 5_000_000  # about 10 per HEALPix pixel at nside 512: no holes in the footprint
    RA_RANGE, DEC_RANGE = (-20.0, 100.0), (-70.0, 0.0)  # deg
    ra = rng.uniform(*RA_RANGE, N_FAKE_STARS) % 360.0
    # Uniform on the sphere: uniform in sin(dec)
    dec = np.degrees(np.arcsin(rng.uniform(*np.sin(np.radians(DEC_RANGE)), N_FAKE_STARS)))
    # DES g and r uniform in colour-magnitude space (no stream), small errors (pass the default cuts)
    g = rng.uniform(17.0, 25.0, N_FAKE_STARS)
    r = g - rng.uniform(-0.5, 1.5, N_FAKE_STARS)
    errors = np.full(N_FAKE_STARS, 0.02)
    stars = pd.DataFrame({
        "ra": ra, "dec": dec,
        "des_yr6_g_obs": g, "des_yr6_r_obs": r, "des_yr6_g_err": errors, "des_yr6_r_err": errors,
    })

In [ ]:
if not USE_FAKE_CATALOG:
    stars = get_eucliddes_stars()

NSIDE_FOOTPRINT = 512
footprint = survey_footprint_healpix_mask(stars["ra"], stars["dec"], NSIDE_FOOTPRINT, nest=False)
area = (~footprint).sum() * hp.nside2pixarea(NSIDE_FOOTPRINT, degrees=True)
print(f"{len(stars):,} stars; footprint: {area:,.0f} deg2")

## 2. Known streams in the footprint

The galstreams tracks whose part in the footprint, at dm > 14, is at least 1 deg long. The Ibata et al. 2024 tracks, without distances in galstreams, are left out (`add_ibata2024=True` to add them). Loading galstreams is slow: it is done once.

In [ ]:
mws = load_galstreams()
selection = select_galstreams_in_footprint(mws, footprint, nest=False)
selection

In [ ]:
fig, ax = plot_galstreams_selection(selection, mws, footprint, nest=False)

## 3. One stream: frame, window and isochrones

- The frame is the galstreams frame of the track: a gala `GreatCircleICRSFrame`.
- The window covers the track, with margins.
- The isochrones use the age and [Fe/H] of the stream when they are known (`stream_populations.csv`); otherwise the default isochrone, that of the DES search (Shipp et al. 2018): 13 Gyr, Z = 0.0002. Z is on the scale of the Marigo et al. 2017 isochrones used here. Each isochrone gives its own maps: below, the first one.

Here, ATLAS, a validation stream (Shipp et al. 2018), inside the fake footprint; any track of `selection` works.

In [ ]:
TRACK_NAME = "AAU-ATLAS-L21"
stream_name = selection.set_index("track_name").loc[TRACK_NAME, "stream_name"]

track = mws[TRACK_NAME]
stream_frame = track.stream_frame
window = galstreams_phi12_window(track)
track_phi12 = radec_to_phi12(track.track.ra.deg, track.track.dec.deg, stream_frame)
print(f"{TRACK_NAME} ({stream_name}): window phi1 {window[0]:.1f} to {window[1]:.1f}, phi2 {window[2]:.1f} to {window[3]:.1f} deg")

In [ ]:
isochrones = known_stream_isochrone_parameters(stream_name)  # reads stream_populations.csv
isochrone = isochrones[0]  # one set of maps per isochrone
isochrones

## 4. Map at one distance modulus

The match filter keeps the stars near the isochrone at this distance modulus. They are counted on the phi1/phi2 grid and smoothed. Bins outside the footprint are masked, and the counts are corrected at the footprint edges.

First, the footprint on the phi1/phi2 grid of the window, with the track. The distance modulus of the map is the median one of the track.

In [ ]:
BIN_SIZE_DEG = 0.1
SMOOTHING_DEG = 0.3
MATCH_FILTER = {
    "name": "des_gr",
    "reference_band": ("des_yr6", "g"),
    "color_bands": [("des_yr6", "r")],
}

footprint_phi12 = survey_footprint_phi12_mask(footprint, stream_frame, window, BIN_SIZE_DEG, nest=False)
dm_track = float(np.nanmedian(galstreams_dm_along_track(track, track_phi12[0])))
print(f"median distance modulus of the track: {dm_track:.2f}")

phi1_centres, phi2_centres = phi12_bin_centres(window, BIN_SIZE_DEG)
half_bin = BIN_SIZE_DEG / 2
extent = (phi1_centres[0] - half_bin, phi1_centres[-1] + half_bin, phi2_centres[0] - half_bin, phi2_centres[-1] + half_bin)
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(~footprint_phi12.T, origin="lower", extent=extent, aspect="auto", cmap="Greys", vmin=0, vmax=2)
ax.plot(*track_phi12, color="C1", label=TRACK_NAME)
ax.set(xlim=(extent[1], extent[0]), xlabel=r"$\phi_1$ [deg]", ylabel=r"$\phi_2$ [deg]", title="Footprint (grey) in the stream frame")
ax.legend(loc="upper right")
plt.show()

In [ ]:
result = known_stream_search_single_dm(
    stars, stream_frame, window, dm_track, isochrone, MATCH_FILTER,
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
)
fig, ax = plot_known_stream_search_single_dm(result, track_phi12=track_phi12)

## 5. Scan over distance moduli: figures, video and file of maps

`known_stream_search_dm_scan` gives the maps one distance modulus at a time. Each one is drawn and saved; then the frames make the video, and the maps are saved together in one file. The phi1/phi2 maps are small: keeping them all is fine.

The outputs go to `figures/outputs/` (gitignored).

In [ ]:
isochrone_label = f"age{isochrone['age']:g}_z{isochrone['metallicity']:g}"
output_dir = Path("../figures/outputs/known_stream_search") / TRACK_NAME / isochrone_label
output_dir.mkdir(parents=True, exist_ok=True)
distance_moduli = np.arange(15.0, 22.0 + 0.05, 0.1)

results = []
for dm, res in known_stream_search_dm_scan(
    stars, stream_frame, window, distance_moduli, isochrone, MATCH_FILTER,
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
):
    fig, ax = plot_known_stream_search_single_dm(res, track_phi12=track_phi12)
    fig.savefig(output_dir / f"{TRACK_NAME}_dm{dm:.2f}.png", dpi=110)
    plt.close(fig)
    results.append((dm, res))

frames = str(output_dir / f"{TRACK_NAME}_dm*.png")
make_gif_from_plots(output_dir / f"{TRACK_NAME}.gif", pattern=frames, duration=300)
make_mp4(output_dir / f"{TRACK_NAME}.mp4", pattern=frames, duration=300)
save_known_stream_search_maps(
    results, output_dir / f"{TRACK_NAME}_maps.npz",
    metadata={"track_name": TRACK_NAME, "isochrone": isochrone},
)

## 6. Following the distance of the track

A stream can have a distance gradient. Here the match filter follows the distance modulus of the track along phi1, plus an offset: one map per offset.

In [ ]:
for dm_offset, res in known_stream_search_dm_along_track(
    stars, track, stream_frame, window, isochrone, MATCH_FILTER,
    dm_offsets=(-0.5, 0.0, 0.5),
    bin_size_deg=BIN_SIZE_DEG, smoothing_deg=SMOOTHING_DEG, footprint_mask=footprint_phi12,
):
    fig, ax = plot_known_stream_search_single_dm(
        res, track_phi12=track_phi12, title=f"{TRACK_NAME}, dm offset {dm_offset:+.1f}"
    )

## 7. In one call, for one stream or for all of them

The workflows do steps 3 to 5, for each isochrone of the stream, from a configuration (`configs/known_stream_search.yaml`): for one track, or for every track crossing the footprint.

In [ ]:
config = RunConfig.from_yaml("../configs/known_stream_search.yaml")

summary = run_known_stream_search(
    stars, TRACK_NAME, config,
    mws=mws, population_table=population_table, footprint_mask=footprint, nest=False,
    output_dir="../figures/outputs/known_stream_search",
)
summary

In [ ]:
table = run_all_known_stream_searches(
    stars, config, nest=False, output_dir="../figures/outputs/known_stream_search",
    mws=mws, population_table=population_table, footprint_mask=footprint,
)
table